### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behaviour with logging, analytic, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early terminatoin logic.
- Applying rate limits, guradrails, and pll detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dilagues with extensive history.
- Application where preserving full conversation context matters.


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Messages summarization
agent = create_agent(
    model="gpt-5",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [7]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [8]:
#Alternative test data
questions = [
    "what is 2 + 1?",
    "what is 2 * 5?", 
    "what is 30/3?",
    "what is 15 - 6?",
    "what is 4 * 4?",
    "what is 38-3?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='what is 2 + 1?', additional_kwargs={}, response_metadata={}, id='bec705a6-200c-49ba-ac31-2fc58676f3d2'), AIMessage(content='3', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENM0pIHBDeGbPfKaWOSOehE9EijFa', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a096b5-6a52-78d2-9170-c277acf22a47-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input_token_

### Token Size


In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}: 
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model="gpt-5",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("tokens", 550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

#Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token 

In [13]:
#Run tes 
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response["messages"])}")

Paris: ~158 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='ba01c916-8021-4f1c-86c0-fe0af0b9c0aa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 134, 'total_tokens': 350, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENMDCzwb9RF808pbhkE7EiV8tY3id', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a096c1-1cac-7c51-ba89-13a64b5227f9-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_9yrKh9pDSoImcpPEYLZ7xfsN', 

### Fraction

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}: 
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model="gpt-5",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("fraction", 0.005), #0.5% = ~640 tokens
            keep=("fraction",0.002), #0.2% = ~256 tokens
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

#Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token 

#Run tes 
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt - 5 context 
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(response['messages'])

Paris: ~133 tokens (0.1039%), 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='fd815145-862d-432b-be13-f137bfdb459e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 134, 'total_tokens': 350, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENMJAi5Oqol7wjEBcroMFjzcFD8MG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a096c6-c3e4-7450-9754-e8446a6c4c90-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_MODG8U2O9NI7Q4H1h

### Human in the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-states operations requiring human approval(e.g., database writes, financial transactions).
- compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    "Mock fubction to read an email by its ID."
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    "Mock function to send an email."
    return f"Email sent to {recipient} with subject '{subject}'"

In [22]:
agent=create_agent(
    model="gpt-5",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [23]:
config = {"configurable": {"thread_id": "test-approve"}}
#step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send Email to john@test.com with subject 'hello' and body 'How are you?'")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send Email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='79127ddf-f025-452e-b352-2c48f7170097'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 180, 'total_tokens': 281, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENMdqLiEhcWl4FI22aG9Tg6REnUPp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a096da-50bd-7111-8cfa-e53b4e8d99de-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@t

In [25]:
from langgraph.types import Command

#step 2 : Approve
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: Your email has been sent to john@test.com with the subject "hello" and the body "How are you?".


### Reject

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    "Mock fubction to read an email by its ID."
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    "Mock function to send an email."
    return f"Email sent to {recipient} with subject '{subject}'"

agent=create_agent(
    model="gpt-5",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)


In [28]:
config = {"configurable": {"thread_id": "test-reject"}}
#step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send Email to john@test.com with subject 'hello' and body 'How are you?'")]},
    config=config
)

In [29]:
from langgraph.types import Command

#step 2 : Approve
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: I couldn’t send the email because the send action was canceled. Here’s the draft:

- To: john@test.com
- Subject: hello
- Body: How are you?

Would you like me to send it now, or make any changes?


### Editing 

In [36]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    "Mock fubction to read an email by its ID."
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    "Mock function to send an email."
    return f"Email sent to {recipient} with subject '{subject}'"

agent=create_agent(
    model="gpt-5",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)


In [37]:
config = {"configurable": {"thread_id": "test-edit"}}
#step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send Email to john@test.com with subject 'hello' and body 'How are you?'")]},
    config=config
)

In [40]:
from langgraph.types import Command

#step 2 : Approve
if "__interrupt__" in result:
    print("Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "edit",
                     "edited_action": {
                         "name": "send_email_tool",
                         "args": {
                             "recipient": "correct@gmail.com",
                             "subject": "Corrected Subject",
                             "body": "This was edited by human before sedning"
                         }
                     }}
                ]
            }
        ),
        config=config
    )


Paused! Editing...


In [39]:
result

{'messages': [HumanMessage(content="Send Email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b8262594-571a-4be0-aa2e-dee42c0f803b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 180, 'total_tokens': 281, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENMkxSnOLMqGurOYGzSd8O9aG6Ow7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a096e1-0f48-7952-a224-72306f34568a-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': 